## Report
```
The input RNA string (up to 10 kbp) is encoded on the host into an integer array of codon indices (0-63) by mapping each base U/C/A/G to 0/1/2/3
and computing index = b0*16 + b1*4 + b2 for each triplet.
A 64-element codon translation table is stored in __constant__ memory, with stop codons represented as '\0'.
A CUDA kernel assigns one thread per codon and each thread writes the corresponding amino acid character to an output array.
The result is copied back to the host and printed as the translated protein string.
```

In [1]:
%%writefile translating_rna_into_protein.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define RNA_INPUT "AUGGCCAUGGCGCCCAGAACUGAGAUCAAUAGUACCCGUAUUAACGGGUGA"

__constant__ char d_codon_table[64];

__global__ void translating_rna_into_protein_kernel(const int *codons, char *out, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        out[idx] = d_codon_table[codons[idx]];
    }
}

static void encode_rna(const char *s, int *out, int n)
{
    int num_codons = n / 3;

    for (int i = 0; i < num_codons; i++) {
        int idx = 0;

        for (int j = 0; j < 3; j++) {
            int base;

            switch (s[i * 3 + j]) {
                case 'U':
                    base = 0;

                    break;

                case 'C':
                    base = 1;

                    break;

                case 'A':
                    base = 2;

                    break;

                default:
                    base = 3;

                    break;
            }

            idx = idx * 4 + base;
        }

        out[i] = idx;
    }
}

void launch_protein_kernel(const int *h_codons, int num_codons)
{
    static const char h_codon_table[64] = {
        'F', 'F', 'L', 'L',  /* UUU UUC UUA UUG */
        'S', 'S', 'S', 'S',  /* UCU UCC UCA UCG */
        'Y', 'Y', 0, 0,      /* UAU UAC UAA UAG */
        'C', 'C', 0, 'W',    /* UGU UGC UGA UGG */
        'L', 'L', 'L', 'L',  /* CUU CUC CUA CUG */
        'P', 'P', 'P', 'P',  /* CCU CCC CCA CCG */
        'H', 'H', 'Q', 'Q',  /* CAU CAC CAA CAG */
        'R', 'R', 'R', 'R',  /* CGU CGC CGA CGG */
        'I', 'I', 'I', 'M',  /* AUU AUC AUA AUG */
        'T', 'T', 'T', 'T',  /* ACU ACC ACA ACG */
        'N', 'N', 'K', 'K',  /* AAU AAC AAA AAG */
        'S', 'S', 'R', 'R',  /* AGU AGC AGA AGG */
        'V', 'V', 'V', 'V',  /* GUU GUC GUA GUG */
        'A', 'A', 'A', 'A',  /* GCU GCC GCA GCG */
        'D', 'D', 'E', 'E',  /* GAU GAC GAA GAG */
        'G', 'G', 'G', 'G'   /* GGU GGC GGA GGG */
    };

    cudaMemcpyToSymbol(d_codon_table, h_codon_table, 64 * sizeof(char));

    int *d_codons = NULL;
    char *d_out = NULL;

    cudaMalloc((void **)&d_codons, num_codons * sizeof(int));
    cudaMalloc((void **)&d_out, (num_codons + 1) * sizeof(char));

    cudaMemset(d_out, 0, (num_codons + 1) * sizeof(char));
    cudaMemcpy(d_codons, h_codons, num_codons * sizeof(int), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (num_codons + block_size - 1) / block_size;

    translating_rna_into_protein_kernel<<<grid_size, block_size>>>(d_codons, d_out, num_codons);

    cudaDeviceSynchronize();

    char *h_out = (char *)malloc((num_codons + 1) * sizeof(char));

    cudaMemcpy(h_out, d_out, (num_codons + 1) * sizeof(char), cudaMemcpyDeviceToHost);

    printf("%s\n", h_out);

    free(h_out);
    cudaFree(d_codons);
    cudaFree(d_out);
}

int main(void)
{
    const char *rna = RNA_INPUT;
    int n = (int)strlen(rna);
    int num_codons = n / 3;
    int *h_codons = (int *)malloc(num_codons * sizeof(int));

    encode_rna(rna, h_codons, n);

    launch_protein_kernel(h_codons, num_codons);

    free(h_codons);

    return 0;
}

Writing translating_rna_into_protein.cu


In [2]:
!nvcc -arch=sm_75 translating_rna_into_protein.cu -o translating_rna_into_protein

In [3]:
!./translating_rna_into_protein

MAMAPRTEINSTRING
